# Vital Signs Generator

The VitalSignsGenerator creates realistic physiological time series for different patient types (healthy, cardiac, sepsis, respiratory, hypertensive). It supports circadian rhythms, heart rate variability, and clinical events.

In [ ]:
import matplotlib.pyplot as plt

from synforecast.generators import VitalSignsGenerator

## Heart Rate - Healthy Patient

Generate 24 hours of per-minute heart rate data for a healthy patient with circadian rhythm, HRV, and clinical events enabled.

In [ ]:
params = {
    "min_length": 1440,
    "max_length": 1440,
    "freq": "min",
    "patient_type": "healthy",
    "vital_sign": "heart_rate",
    "include_circadian": True,
    "include_hrv": True,
    "include_events": True,
    "seed": 42,
}

generator = VitalSignsGenerator(engine="polars", **params)
df = generator.generate(n_series=1)

values = df["y"].to_numpy()
print(f"Heart rate range: [{values.min():.1f}, {values.max():.1f}] bpm")
print(f"Mean heart rate: {values.mean():.1f} bpm")
print(f"Std deviation: {values.std():.1f} bpm")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["ds"].to_list(), df["y"].to_list(), alpha=0.8, linewidth=0.5)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Heart Rate (bpm)")
ax.set_title("Heart Rate - Healthy Patient (24 Hours)")
plt.tight_layout()
plt.show()

## Comparing Patient Types

Compare heart rate distributions across different patient conditions.

In [ ]:
patient_types = ["healthy", "cardiac", "sepsis", "respiratory", "hypertensive"]

for ptype in patient_types:
    gen = VitalSignsGenerator(engine="polars", 
        **{
            "min_length": 1440,
            "max_length": 1440,
            "freq": "min",
            "patient_type": ptype,
            "vital_sign": "heart_rate",
            "seed": 42,
        }
    )
    df_patient = gen.generate(n_series=1)
    hr = df_patient["y"].to_numpy()
    print(
        f"{ptype:15s}: mean={hr.mean():.1f}, std={hr.std():.1f}, range=[{hr.min():.0f}, {hr.max():.0f}]"
    )

In [ ]:
fig, axes = plt.subplots(len(patient_types), 1, figsize=(12, 2.5 * len(patient_types)), sharex=True)
for ax, ptype in zip(axes, patient_types):
    gen = VitalSignsGenerator(engine="polars", **{"min_length": 1440, "max_length": 1440, "freq": "min", "patient_type": ptype, "vital_sign": "heart_rate", "seed": 42})
    df_p = gen.generate(n_series=1)
    ax.plot(df_p["ds"].to_list(), df_p["y"].to_list(), alpha=0.8, linewidth=0.5)
    ax.set_ylabel("HR (bpm)")
    ax.set_title(ptype)
plt.xlabel("Timestamp")
plt.suptitle("Heart Rate by Patient Type")
plt.tight_layout()
plt.show()

## All Vital Signs - Sepsis Patient

Generate all six vital signs simultaneously for a sepsis patient over 8 hours.

In [ ]:
sepsis_gen = VitalSignsGenerator(engine="polars", 
    **{
        "min_length": 480,
        "max_length": 480,
        "freq": "min",
        "patient_type": "sepsis",
        "seed": 42,
    }
)
all_vitals_df = sepsis_gen.generate_all_vitals(n_series=1)

print("Vital sign statistics for sepsis patient:")
vital_cols = [
    "heart_rate",
    "systolic_bp",
    "diastolic_bp",
    "respiratory_rate",
    "spo2",
    "temperature",
]

for col in vital_cols:
    vals = all_vitals_df[col].to_numpy()
    print(
        f"  {col:18s}: mean={vals.mean():.1f}, range=[{vals.min():.1f}, {vals.max():.1f}]"
    )

In [ ]:
vital_cols = ["heart_rate", "systolic_bp", "diastolic_bp", "respiratory_rate", "spo2", "temperature"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, vital_cols):
    ax.plot(all_vitals_df["ds"].to_list(), all_vitals_df[col].to_list(), alpha=0.8, linewidth=0.5)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Timestamp")
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("All Vital Signs - Sepsis Patient (8 Hours)", fontsize=14)
plt.tight_layout()
plt.show()

## Oxygen Saturation (SpO2) Comparison

Compare SpO2 levels across healthy, respiratory, and sepsis patients. Lower SpO2 and more time below 95% indicates worse oxygenation.

In [ ]:
for ptype in ["healthy", "respiratory", "sepsis"]:
    gen = VitalSignsGenerator(engine="polars", 
        **{
            "min_length": 1440,
            "max_length": 1440,
            "freq": "min",
            "patient_type": ptype,
            "vital_sign": "spo2",
            "seed": 42,
        }
    )
    df_spo2 = gen.generate(n_series=1)
    spo2 = df_spo2["y"].to_numpy()
    below_95 = (spo2 < 95).sum() / len(spo2) * 100
    print(
        f"{ptype:12s}: mean={spo2.mean():.1f}%, min={spo2.min():.1f}%, time <95%: {below_95:.1f}%"
    )

## Model Information

Inspect the generator configuration and baseline values for each vital sign.

In [ ]:
info = generator.get_model_info()
print(f"Patient type: {info['patient_type']}")
print(f"Current vital sign: {info['vital_sign']}")
print(f"Circadian rhythm: {info['include_circadian']}")
print(f"HRV included: {info['include_hrv']}")

print("\nBaseline values for healthy patient:")
for vital, params in info["baselines"].items():
    print(
        f"  {vital}: mean={params['mean']}, range=[{params['min']}, {params['max']}]"
    )